In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

!pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("/content/ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading PDF document...
Document Loaded
Total Characters: 2812

Preview:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub

Splitting document into chunks...
Total Chunks Created: 7

Example Chunk:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applicat

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: What is artifical Intelligence


Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub agents.
Machine Learning
Machine Learning is a subset of artificial intelligence that focuses on algorithms that improve
automatically through experience. Instead of being explicitly programmed, machine learning models
learn patterns from data. Common types of machine learning include supervised learning,
unsupervised learning, and reinforce

In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.



In [ ]:
print("Loading PDF document...")

reader = PdfReader("/content/data_privacy_regulation.pdf")

legal_text = ""

# Extract text from every page
for page in reader.pages:
    legal_text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(legal_text))

# Preview some text
print("\nPreview:\n")
print(legal_text[:500])

Loading PDF document...
Document Loaded
Total Characters: 2885

Preview:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr


In [ ]:
print("\nSplitting legal document into chunks...")

legal_chunks = chunk_text(legal_text, chunk_size=500, overlap=50)

print("Total Chunks Created:", len(legal_chunks))

print("\nExample Chunk:\n")
print(legal_chunks[0])


Splitting legal document into chunks...
Total Chunks Created: 7

Example Chunk:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr


In [ ]:
print("\nCreating embeddings for legal chunks...")

# Create embedding for the first legal chunk as an example
example_legal_embedding = embedding_model.encode(legal_chunks[0]).tolist()

print("Example embedding created.")
print("Embedding dimension:", len(example_legal_embedding))


Creating embeddings for legal chunks...
Example embedding created.
Embedding dimension: 384


In [ ]:
print("\nInitializing ChromaDB client and preparing collection...")

client = chromadb.Client()

# Delete collection if it exists to ensure a clean start for the legal document
try:
    client.delete_collection("legal_pdf_collection")
    print("Old 'legal_pdf_collection' deleted.")
except:
    print("No old 'legal_pdf_collection' found or error during deletion.")

collection = client.create_collection("legal_pdf_collection")

print("New 'legal_pdf_collection' created.")


Initializing ChromaDB client and preparing collection...
No old 'legal_pdf_collection' found or error during deletion.
New 'legal_pdf_collection' created.


In [ ]:
print("\nCreating embeddings and storing legal chunks in ChromaDB...")

for i, chunk in enumerate(legal_chunks):

    # Create embedding vector for the legal chunk
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All legal chunks stored successfully in 'legal_pdf_collection'.")


Creating embeddings and storing legal chunks in ChromaDB...
All legal chunks stored successfully in 'legal_pdf_collection'.


In [ ]:
print("\nDefining the legal document retriever function...")

def retrieve_legal_docs(query, k=3):
    """
    Converts user query to embedding and finds most similar legal document chunks
    """

    # Convert query to embedding using the pre-loaded embedding model
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search on the legal_pdf_collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]

print("Legal document retriever function defined.")

# Test the retriever with an example query
example_query = "What are the rules for cross-border data transfers?"
print(f"\nTesting retriever with query: '{example_query}'")
retrieved_docs = retrieve_legal_docs(example_query)
print("Retrieved Documents:")
for doc in retrieved_docs:
    print("-", doc[:100] + "...") # Print first 100 characters of each retrieved doc


Defining the legal document retriever function...
Legal document retriever function defined.

Testing retriever with query: 'What are the rules for cross-border data transfers?'
Retrieved Documents:
- ta.

Individuals have the right to request deletion of personal data under certain conditions.

In...
-  binding corporate rules, or regulatory approvals before
transferring personal data internationally....
- Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educ...


In [ ]:
print("\nLoading LLM for legal document Q&A...")

legal_qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM for legal document Q&A loaded successfully")


Loading LLM for legal document Q&A...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM for legal document Q&A loaded successfully


In [ ]:
from transformers import pipeline

print("\nLoading LLM for legal document Q&A...")

legal_qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM for legal document Q&A loaded successfully")


Loading LLM for legal document Q&A...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM for legal document Q&A loaded successfully


In [ ]:
print("\nLoading LLM for legal document Q&A...")

legal_qa_pipeline = pipeline(
    "question-answering",
    model="google/flan-t5-base"
)

print("LLM for legal document Q&A loaded successfully")


Loading LLM for legal document Q&A...


Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

T5ForQuestionAnswering LOAD REPORT from: google/flan-t5-base
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
qa_outputs.weight | MISSING    | 
qa_outputs.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LLM for legal document Q&A loaded successfully


In [ ]:
print("\nLoading LLM for legal document Q&A...")

legal_qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM for legal document Q&A loaded successfully")


Loading LLM for legal document Q&A...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM for legal document Q&A loaded successfully


In [ ]:
print("\nDefining the legal document question-answering function...")

def answer_legal_question(query):

    # Retrieve relevant context using the legal retriever
    context_docs = retrieve_legal_docs(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = legal_qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]

print("Legal document question-answering function defined.")

# Test the Q&A function with an example question
example_legal_question = "What are the main penalties for non-compliance with this regulation?"
print(f"\nTesting Q&A with question: '{example_legal_question}'")
legal_answer = answer_legal_question(example_legal_question)
print("Answer:\n", legal_answer)


Defining the legal document question-answering function...
Legal document question-answering function defined.

Testing Q&A with question: 'What are the main penalties for non-compliance with this regulation?'


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
es for Non■Compliance
Organizations found to be in violation of this regulation may face financial penalties, regulatory
investigations, and operational restrictions. Administrative fines may reach up to 4 percent of the
company's annual global revenue or a fixed penalty determined by the regulatory authority.
Repeated violations may result in suspension of data processing activities.
Article 7: Compliance and Audits
Organizations must maintain documentation demonstrating compliance with this re  binding corporate rules, or regulatory approvals before
transferring personal data internationally.
Article 5: Data Security and Breach Notification

Organizations must implement encryption, access control, and monitoring systems.

Any data breach that risks individual privacy must be reported

In [ ]:
print("\nDefining the legal document question-answering function...")

def answer_legal_question(query):

    # Retrieve relevant context using the legal retriever
    context_docs = retrieve_legal_docs(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = legal_qa_pipeline(
        prompt,
        max_new_tokens=256, # Changed from max_length to max_new_tokens
        temperature=0.5
    )

    return response[0]["generated_text"]

print("Legal document question-answering function defined.")

# Test the Q&A function with an example question
example_legal_question = "What are the main penalties for non-compliance with this regulation?"
print(f"\nTesting Q&A with question: '{example_legal_question}'")
legal_answer = answer_legal_question(example_legal_question)
print("Answer:\n", legal_answer)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Defining the legal document question-answering function...
Legal document question-answering function defined.

Testing Q&A with question: 'What are the main penalties for non-compliance with this regulation?'


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
es for Non■Compliance
Organizations found to be in violation of this regulation may face financial penalties, regulatory
investigations, and operational restrictions. Administrative fines may reach up to 4 percent of the
company's annual global revenue or a fixed penalty determined by the regulatory authority.
Repeated violations may result in suspension of data processing activities.
Article 7: Compliance and Audits
Organizations must maintain documentation demonstrating compliance with this re  binding corporate rules, or regulatory approvals before
transferring personal data internationally.
Article 5: Data Security and Breach Notification

Organizations must implement encryption, access control, and monitoring systems.

Any data breach that risks individual privacy must be reported

In [ ]:
print("\n==============================")
print("Legal RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

# Predefined questions
default_questions = [
    "What does the regulation say about cross-border data transfer?",
    "What penalties are mentioned for non-compliance?",
    "What are the responsibilities of data controllers?",
    "Does this regulation conflict with GDPR?",
    "What rules apply to personal data protection?"
]

# First show answers for predefined questions
for q in default_questions:
    print("Ask a question:", q)

    legal_answer = answer_legal_question(q)

    print("\nAnswer:\n", legal_answer)
    print("\n" + "-"*60 + "\n")


# Then allow user input questions
while True:

    legal_question = input("Ask a question: ")

    if legal_question.lower() == "exit":
        print("Goodbye!")
        break

    legal_answer = answer_legal_question(legal_question)

    print("\nAnswer:\n", legal_answer)
    print("\n" + "-"*60 + "\n")


Legal RAG Chatbot Ready
Type 'exit' to stop

Ask a question: What does the regulation say about cross-border data transfer?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
ta.

Individuals have the right to request deletion of personal data under certain conditions.

Individuals may object to automated decision■making processes involving their data.
Article 4: Cross■Border Data Transfers
Transfer of personal data to another country or jurisdiction is permitted only when the receiving
country ensures an adequate level of data protection. Organizations must implement safeguards
such as standard contractual clauses, binding corporate rules, or regulatory approvals   binding corporate rules, or regulatory approvals before
transferring personal data internationally.
Article 5: Data Security and Breach Notification

Organizations must implement encryption, access control, and monitoring systems.

Any data breach that risks individual privacy must be reporte

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
es for Non■Compliance
Organizations found to be in violation of this regulation may face financial penalties, regulatory
investigations, and operational restrictions. Administrative fines may reach up to 4 percent of the
company's annual global revenue or a fixed penalty determined by the regulatory authority.
Repeated violations may result in suspension of data processing activities.
Article 7: Compliance and Audits
Organizations must maintain documentation demonstrating compliance with this re  binding corporate rules, or regulatory approvals before
transferring personal data internationally.
Article 5: Data Security and Breach Notification

Organizations must implement encryption, access control, and monitoring systems.

Any data breach that risks individual privacy must be reporte

...


Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr ta.

Individuals have the right to request deletion of personal data under certain conditions.

Individuals may object to automated decision■making processes involving their data.
Article 4: Cross■Border Data Transfers
Transfer of personal data to another country or jurisdiction is permitted o

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr ing personal data.

Personal data should be collected for specified, explicit, and legitimate purposes.

Data minimization principles must be followed, ensuring only necessary data is collected.

Organizations must implement appropriate technical and organizational security measures.
Article 

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI legal assistant.

Answer the legal question using ONLY the context below.
If the answer is not present in the provided document, say "Not found in the legal document".

Context:
Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr ing personal data.

Personal data should be collected for specified, explicit, and legitimate purposes.

Data minimization principles must be followed, ensuring only necessary data is collected.

Organizations must implement appropriate technical and organizational security measures.
Article 

In [ ]:
# Scenario: University Library Assistant
# A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
# How It Works
# - Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
# - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
# - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
# - Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
# - LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
# - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.


In [ ]:
import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [ ]:
print("Loading PDF document...")

reader = PdfReader("/content/Introduction_to_Data_Science.pdf")

textbook_text = ""

# Extract text from every page
for page in reader.pages:
    textbook_text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(textbook_text))

# Preview some text
print("\nPreview:\n")
print(textbook_text[:500])

Loading PDF document...
Document Loaded
Total Characters: 3427

Preview:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing, 


In [ ]:
print("Creating dummy 'Introduction_to_Data_Science.pdf'...")

# Install reportlab if not already installed
!pip install reportlab

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

def create_dummy_pdf(filename="/content/Introduction_to_Data_Science.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "Introduction to Data Science")
    c.drawString(100, 730, "Chapter 1: What is Data Science?")
    c.drawString(120, 710, "Data Science is an interdisciplinary field that uses scientific methods, processes,")
    c.drawString(120, 695, "algorithms and systems to extract knowledge and insights from noisy, structured and")
    c.drawString(120, 680, "unstructured data.")
    c.drawString(100, 660, "Chapter 2: Machine Learning Fundamentals")
    c.drawString(120, 640, "Machine Learning (ML) is a subset of artificial intelligence that focuses on")
    c.drawString(120, 625, "the development of algorithms that allow computers to learn from data without being")
    c.drawString(120, 610, "explicitly programmed.")
    c.drawString(100, 590, "Key concepts include supervised learning, unsupervised learning, and reinforcement learning.")
    c.save()
    print(f"Dummy PDF '{filename}' created successfully.")

create_dummy_pdf()

print("Loading PDF document...")

reader = PdfReader("/content/Introduction_to_Data_Science.pdf")

textbook_text = ""

# Extract text from every page
for page in reader.pages:
    textbook_text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(textbook_text))

# Preview some text
print("\nPreview:\n")
print(textbook_text[:500])


Creating dummy 'Introduction_to_Data_Science.pdf'...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.5 MB/s eta 0:00:00
Dummy PDF '/content/Introduction_to_Data_Science.pdf' created successfully.
Loading PDF document...
Document Loaded
Total Characters: 567

Preview:

Introduction to Data Science
Chapter 1: What is Data Science?
Data Science is an interdisciplinary field that uses scientific methods, processes,
algorithms and systems to extract knowledge and insights from noisy, structured and
unstructured data.
Chapter 2: Machine Learning Fundamentals
Machine Learning (ML) is a subset of artificial intelligence that focuses on
the development of algorithms that allow computers to learn from data without being
explicitly programmed.
Key concepts include super


In [ ]:
print("\nSplitting textbook document into chunks...")

textbook_chunks = chunk_text(textbook_text, chunk_size=500, overlap=50)

print("Total Chunks Created:", len(textbook_chunks))

print("\nExample Chunk:\n")
print(textbook_chunks[0])


Splitting textbook document into chunks...
Total Chunks Created: 2

Example Chunk:

Introduction to Data Science
Chapter 1: What is Data Science?
Data Science is an interdisciplinary field that uses scientific methods, processes,
algorithms and systems to extract knowledge and insights from noisy, structured and
unstructured data.
Chapter 2: Machine Learning Fundamentals
Machine Learning (ML) is a subset of artificial intelligence that focuses on
the development of algorithms that allow computers to learn from data without being
explicitly programmed.
Key concepts include super


In [ ]:
print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [ ]:
print("\nInitializing ChromaDB client and preparing collection for textbook...")

client = chromadb.Client()

# Delete collection if it exists to ensure a clean start for the textbook document
try:
    client.delete_collection("textbook_collection")
    print("Old 'textbook_collection' deleted.")
except:
    print("No old 'textbook_collection' found or error during deletion.")

collection = client.create_collection("textbook_collection")

print("New 'textbook_collection' created.")


Initializing ChromaDB client and preparing collection for textbook...
No old 'textbook_collection' found or error during deletion.
New 'textbook_collection' created.


In [ ]:
print("\nCreating embeddings and storing textbook chunks in ChromaDB...")

for i, chunk in enumerate(textbook_chunks):

    # Create embedding vector for the textbook chunk
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All textbook chunks stored successfully in 'textbook_collection'.")


Creating embeddings and storing textbook chunks in ChromaDB...
All textbook chunks stored successfully in 'textbook_collection'.


In [ ]:
print("\nDefining the textbook document retriever function...")

def retrieve_textbook_docs(query, k=3):
    """
    Converts user query to embedding and finds most similar textbook document chunks
    """

    # Convert query to embedding using the pre-loaded embedding model
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search on the textbook_collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]

print("Textbook document retriever function defined.")

# Test the retriever with an example query
example_textbook_query = "What is machine learning?"
print(f"\nTesting retriever with query: '{example_textbook_query}'")
retrieved_textbook_docs = retrieve_textbook_docs(example_textbook_query)
print("Retrieved Documents:")
for doc in retrieved_textbook_docs:
    print("-", doc[:100] + "...") # Print first 100 characters of each retrieved doc


Defining the textbook document retriever function...
Textbook document retriever function defined.

Testing retriever with query: 'What is machine learning?'
Retrieved Documents:
- Introduction to Data Science
Chapter 1: What is Data Science?
Data Science is an interdisciplinary f...
- 
explicitly programmed.
Key concepts include supervised learning, unsupervised learning, and reinfor...


In [ ]:
print("\nLoading LLM for textbook Q&A...")

textbook_qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM for textbook Q&A loaded successfully")


Loading LLM for textbook Q&A...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM for textbook Q&A loaded successfully


In [ ]:
print("\nLoading LLM for textbook Q&A...")

textbook_qa_pipeline = pipeline(
    "question-answering",
    model="google/flan-t5-base"
)

print("LLM for textbook Q&A loaded successfully")


Loading LLM for textbook Q&A...


Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

T5ForQuestionAnswering LOAD REPORT from: google/flan-t5-base
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
qa_outputs.weight | MISSING    | 
qa_outputs.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LLM for textbook Q&A loaded successfully


In [ ]:
print("\nDefining the textbook question-answering function...")

def answer_textbook_question(query):

    # Retrieve relevant context using the textbook retriever
    context_docs = retrieve_textbook_docs(query)

    context = " ".join(context_docs)

    # For question-answering pipeline, we typically provide a question and a context
    response = textbook_qa_pipeline(
        question=query,
        context=context
    )

    return response["answer"]

print("Textbook question-answering function defined.")

# Test the Q&A function with an example question
example_textbook_question = "What is the definition of Data Science?"
print(f"\nTesting Q&A with question: '{example_textbook_question}'")
textbook_answer = answer_textbook_question(example_textbook_question)
print("Answer:\n", textbook_answer)


Defining the textbook question-answering function...
Textbook question-answering function defined.

Testing Q&A with question: 'What is the definition of Data Science?'
Answer:
 unsupervised learning,


In [ ]:
print("\n==============================")
print("Textbook RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    textbook_question = input("Ask a question about the textbook: ")

    if textbook_question.lower() == "exit":
        print("Goodbye!")
        break

    textbook_answer = answer_textbook_question(textbook_question)

    print("\nAnswer:\n", textbook_answer)
    print("\n" + "-"*60 + "\n")


Textbook RAG Chatbot Ready
Type 'exit' to stop

Ask a question about the textbook: which textbook of science can i used

Answer:
 unsupervised learning,

------------------------------------------------------------

Ask a question about the textbook: What is Data Science

Answer:
 unsupervised learning,

------------------------------------------------------------

Ask a question about the textbook: exit
Goodbye!


In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://484f24aa57053519f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==========================================================
# HEALTHCARE POLICY NAVIGATOR
# GRADIO + CHROMA + EMBEDDINGS + HUGGINGFACE LLM
# ==========================================================

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 1 — Load Language Model
# ----------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM ready")


# ----------------------------------------------------------
# STEP 2 — Load Embedding Model
# ----------------------------------------------------------

print("Loading embeddings...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")


# ----------------------------------------------------------
# STEP 3 — Initialize Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("policy_docs")
except:
    pass

collection = client.create_collection("policy_docs")


# ----------------------------------------------------------
# STEP 4 — Chunking Function
# ----------------------------------------------------------

def chunk_text(text, chunk_size=600, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# ----------------------------------------------------------
# STEP 5 — Process Uploaded Policy PDF
# ----------------------------------------------------------

def process_pdf(file):

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Policy document processed successfully. {len(chunks)} sections indexed."


# ----------------------------------------------------------
# STEP 6 — Retriever
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 7 — Generate Compliance Answer
# ----------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a Healthcare Policy Navigator.

Use the policy context below to answer the question.

Provide a clear and actionable answer that hospital staff
(doctors, administrators, or IT teams) can follow.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm(
        prompt,
        max_length=220,
        temperature=0.2
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 8 — Chat Function
# ----------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a compliance question."

    return answer_question(question)


# ----------------------------------------------------------
# STEP 9 — Gradio Interface
# ----------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Healthcare Policy Navigator")

    gr.Markdown("""
Upload a healthcare regulation document and ask questions about:

• Patient data privacy
• Telemedicine rules
• Compliance requirements
• Penalties for violations
""")

    pdf_input = gr.File(label="Upload Healthcare Regulation PDF")

    upload_button = gr.Button("Process Policy Document")

    status = gr.Textbox(label="System Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Policy Guidance",
        lines=12
    )

    ask_button = gr.Button("Get Guidance")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# ----------------------------------------------------------
# STEP 10 — Launch App
# ----------------------------------------------------------

demo.launch()

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM ready
Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f33719f8e516d25b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [43]:
# ==========================================================
# ENVIRONMENTAL POLICY COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + HUGGINGFACE LLM
# ==========================================================

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 1 — Load Language Model
# ----------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 2 — Load Embedding Model
# ----------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 3 — Initialize Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("environmental_policy_docs")
except:
    pass

collection = client.create_collection("environmental_policy_docs")


# ----------------------------------------------------------
# STEP 4 — Chunking Function
# ----------------------------------------------------------

def chunk_text(text, chunk_size=600, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# ----------------------------------------------------------
# STEP 5 — Process Uploaded Regulation PDF
# ----------------------------------------------------------

def process_pdf(file):

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Environmental regulation processed. {len(chunks)} sections indexed."


# ----------------------------------------------------------
# STEP 6 — Retriever
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 7 — Generate Compliance Answer
# ----------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are an Environmental Policy Compliance Assistant.

Use ONLY the regulation context below to answer the question.

Provide clear and actionable guidance for engineers,
sustainability officers, and executives.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm(
        prompt,
        max_length=220,
        temperature=0.2
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 8 — Chat Function
# ----------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a compliance question."

    return answer_question(question)


# ----------------------------------------------------------
# STEP 9 — Gradio Interface
# ----------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🌍 Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload an environmental regulation document and ask questions about:

• Carbon emission limits
• Waste disposal rules
• Renewable energy targets
• Environmental compliance penalties
""")

    pdf_input = gr.File(label="Upload Environmental Regulation PDF")

    upload_button = gr.Button("Process Regulation Document")

    status = gr.Textbox(label="System Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Compliance Guidance",
        lines=12
    )

    ask_button = gr.Button("Get Answer")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# ----------------------------------------------------------
# STEP 10 — Launch App
# ----------------------------------------------------------

demo.launch()

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31145786f580815122.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
